# nb12 · Generate the poster's audio A/B demo

Produces the clips the poster QR links to: **zero-shot vs finetuned** OmniVoice saying the Yorùbá
tone minimal pairs (`ọkọ́` hoe / `ọkọ̀` vehicle), plus a **PSOLA tone-destroyed** clip (words intact,
tune flattened) — the thing CER can't hear. Reuses the proven nb04 loading/generate code.

**Runtime:** one GPU (T4/L4/A100). **Secrets:** `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `HF_TOKEN`.
Output: `demo_audio/*.wav` (downloaded as a zip) → drop into `docs/audio/` and enable GitHub Pages.

In [ ]:
%pip install --quiet "numpy<2"
%pip install --quiet omnivoice
%pip install --quiet boto3 soundfile soxr librosa "huggingface_hub>=0.24.0" "git+https://github.com/mosesdaudu001/tone-on-a-budget.git"
%pip uninstall -y hf-xet

## 1 · Secrets, S3, and the OmniVoice base snapshot (zero-shot model source)

In [ ]:
import os, getpass, boto3, torch, numpy as np, shutil
def _secret(k):
    try:
        from google.colab import userdata
        v=userdata.get(k)
        if v: return v
    except Exception: pass
    return os.environ.get(k) or getpass.getpass(f"{k}: ")
os.environ["AWS_ACCESS_KEY_ID"]=_secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"]=_secret("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"]=os.environ.get("AWS_DEFAULT_REGION","us-east-1")
HF_TOKEN=_secret("HF_TOKEN"); os.environ["HF_TOKEN"]=HF_TOKEN or ""
if HF_TOKEN:
    from huggingface_hub import login; login(token=HF_TOKEN)
os.environ["HF_HUB_DISABLE_XET"]="1"; os.environ["HF_HUB_ENABLE_HF_TRANSFER"]="0"

BUCKET="codec-audio-data"
s3=boto3.client("s3",region_name=os.environ["AWS_DEFAULT_REGION"]); s3.head_bucket(Bucket=BUCKET)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
SR=24000; LANG_CODE="yo"
CKPT_PREFIX="tts_checkpoints/omnivoice_yoruba/15h"     # the headline finetune; change to .../full for 28.5h
BIBLE_MANIFEST="tts_data/yoruba_gold/s1_train.v2.jsonl"
WORK="/content/ab_demo" if os.path.isdir("/content") else "/tmp/ab_demo"; os.makedirs(WORK,exist_ok=True)
OUT=os.path.join(WORK,"demo_audio"); os.makedirs(OUT,exist_ok=True)

from huggingface_hub import snapshot_download
OV_BASE=None
for _ in range(3):
    try: OV_BASE=snapshot_download("k2-fsa/OmniVoice",max_workers=1,etag_timeout=60); break
    except Exception as e: print("base prefetch retry:",type(e).__name__,e)
assert OV_BASE and os.path.isdir(os.path.join(OV_BASE,"audio_tokenizer")), "could not pre-fetch k2-fsa/OmniVoice"
print("base snapshot:",OV_BASE,"| DEVICE",DEVICE)

## 2 · Materialize the finetuned checkpoint from S3 (same as nb04)

In [ ]:
def s3_ls(prefix):
    out,tok=[],None
    while True:
        kw=dict(Bucket=BUCKET,Prefix=prefix)
        if tok: kw["ContinuationToken"]=tok
        r=s3.list_objects_v2(**kw); out+=[(o["Key"],o["Size"]) for o in r.get("Contents",[])]
        if r.get("IsTruncated"): tok=r.get("NextContinuationToken")
        else: break
    return out
AUX=["tokenizer.json","tokenizer_config.json","special_tokens_map.json","vocab.json",
     "merges.txt","chat_template.jinja","generation_config.json","added_tokens.json"]
def materialize_ckpt(local_dir):
    have=set(os.listdir(local_dir))
    for fn in AUX:
        if fn not in have and os.path.exists(os.path.join(OV_BASE,fn)):
            shutil.copy(os.path.join(OV_BASE,fn),os.path.join(local_dir,fn))
    at=os.path.join(local_dir,"audio_tokenizer")
    if not os.path.isdir(at): shutil.copytree(os.path.join(OV_BASE,"audio_tokenizer"),at)
    return local_dir
def pull_top_level(prefix,dest):
    os.makedirs(dest,exist_ok=True); n=0
    for k,sz in s3_ls(prefix+"/"):
        tail=k[len(prefix)+1:]
        if not tail or "/" in tail: continue
        s3.download_file(BUCKET,k,os.path.join(dest,os.path.basename(k))); n+=1
    return n

CKPT_LOCAL=os.path.join(WORK,"ckpt_ft")
assert s3_ls(CKPT_PREFIX+"/"), f"nothing under s3://{BUCKET}/{CKPT_PREFIX}/"
print("downloaded",pull_top_level(CKPT_PREFIX,CKPT_LOCAL),"files ->",CKPT_LOCAL)
materialize_ckpt(CKPT_LOCAL)

## 3 · One reference voice (the clean BibleTTS speaker the finetune used)

In [ ]:
import io, json, soundfile as sf
def _load_ref(s3key):
    lp=f"{WORK}/ref_{os.path.basename(str(s3key))}"
    s3.download_file(BUCKET,s3key,lp)
    w,sr=sf.read(lp,dtype="float32"); w=w.mean(-1) if w.ndim>1 else w
    return lp,w,sr
REF_PATH=REF_TEXT=None
for raw in io.TextIOWrapper(s3.get_object(Bucket=BUCKET,Key=BIBLE_MANIFEST)["Body"],encoding="utf-8"):
    try: r=json.loads(raw)
    except Exception: continue
    if r.get("source")!="bible": continue
    key=r.get("audio_s3_key") or r.get("ref_audio_key") or r.get("wav_key")
    if not (key and r.get("text")): continue
    if not (3.0<=float(r.get("duration_sec",6.0))<=12.0): continue
    REF_PATH,_,_=_load_ref(key); REF_TEXT=r["text"]; break
assert REF_PATH, "no usable bible reference found in the manifest"
print("reference voice:",REF_PATH,"|",REF_TEXT[:60])

## 4 · Generate the A/B clips + the PSOLA tone-destroyed clip

Carrier: *Mo rí ___ ní ọjà* (I saw a ___ at the market). Same reference voice for both models, so the
only difference you hear is the model. Both clips play inline.

In [ ]:
import gc, numpy as np, soundfile as sf, IPython.display as ipd
from omnivoice import OmniVoice
from tone_metric import psola_flatten

PAIRS=[("hoe","Mo rí ọkọ́ ní ọjà."),
       ("vehicle","Mo rí ọkọ̀ ní ọjà.")]      # ọkọ́ = hoe (High) ; ọkọ̀ = vehicle (Low)

def synth(ov, text):
    a=ov.generate(text=text,language=LANG_CODE,ref_audio=REF_PATH,ref_text=REF_TEXT)
    w=a[0] if isinstance(a,(list,tuple)) else a
    return np.asarray(w,dtype="float32").reshape(-1)

def load_ov(path):
    return OmniVoice.from_pretrained(path,device_map=("cuda:0" if DEVICE=="cuda" else "cpu"),dtype=torch.float16)

# zero-shot
ov=load_ov(OV_BASE)
for name,text in PAIRS:
    w=synth(ov,text); sf.write(f"{OUT}/zs_{name}.wav",w,SR); print("zs",name); ipd.display(ipd.Audio(w,rate=SR))
del ov; gc.collect(); torch.cuda.empty_cache() if DEVICE=="cuda" else None

# finetuned (15h)
ov=load_ov(CKPT_LOCAL)
ft={}
for name,text in PAIRS:
    w=synth(ov,text); ft[name]=w; sf.write(f"{OUT}/ft_{name}.wav",w,SR); print("ft",name); ipd.display(ipd.Audio(w,rate=SR))
del ov; gc.collect(); torch.cuda.empty_cache() if DEVICE=="cuda" else None

# PSOLA: destroy the tune, keep the words (the clip CER scores as fine but the tone is wrong)
flat=psola_flatten(ft["vehicle"],SR)
sf.write(f"{OUT}/ft_vehicle_flat.wav",np.asarray(flat,dtype="float32"),SR)
print("ft_vehicle_flat (pitch-flattened)"); ipd.display(ipd.Audio(flat,rate=SR))
print("\nwrote:",sorted(os.listdir(OUT)))

## 5 · Publish

1. Download the clips:

In [ ]:
import shutil
zipp=shutil.make_archive(os.path.join(WORK,"demo_audio"),"zip",OUT)
print("zip:",zipp)
try:
    from google.colab import files; files.download(zipp)
except Exception: print("(not on Colab — grab", zipp, "manually)")

2. Unzip into **`docs/audio/`** in the `tone-on-a-budget` repo (filenames must match `docs/index.html`:
   `zs_hoe.wav`, `ft_hoe.wav`, `zs_vehicle.wav`, `ft_vehicle.wav`, `ft_vehicle_flat.wav`).
3. `git add docs && git commit -m 'audio demo' && git push`.
4. On GitHub: **Settings → Pages → Deploy from branch → `main` / `/docs`**. The page goes live at
   `https://mosesdaudu001.github.io/tone-on-a-budget/` — exactly where the poster QR points.